In [9]:
from label_studio_sdk.label_interface import LabelInterface
from label_studio_sdk.label_interface.create import labels
from label_studio_sdk.actions import ActionsCreateRequestSelectedItemsExcluded
from ls_helper import *

In [ ]:
# Define the URL where Label Studio is accessible and the API key for your user account
LABEL_STUDIO_URL = 'http://localhost:8080'
LABEL_STUDIO_ML_BACKEND_URL = 'http://localhost:9090'

# API key is available at the Account & Settings > Access Tokens page in Label Studio UI
API_KEY = '<your api key>'

# Import the SDK and the client module
from label_studio_sdk.client import LabelStudio

# Connect to the Label Studio API and check the connection
ls = LabelStudio(base_url=LABEL_STUDIO_URL, api_key=API_KEY)

In [11]:
# Define labeling interface
label_config = LabelInterface.create({
    'text': 'Text',
    'label': labels(['PER', 'ORG', 'LOC', 'MISC'])
})

FLAIR_TEST_PROJECT = 'flair-ner-test'
RECREATE_PROJECT = False

In [12]:
# Create a project with the specified title and labeling configuration
project = get_or_create_project(ls, FLAIR_TEST_PROJECT, label_config, RECREATE_PROJECT)
print(project)

id=50 title='flair-ner-test' description='' label_config='<View>\n  <Text name="text" value="$text"/>\n  <Labels name="label" toName="text">\n    <Label value="PER"/>\n    <Label value="ORG"/>\n    <Label value="LOC"/>\n    <Label value="MISC"/>\n  </Labels>\n</View>' expert_instruction='' show_instruction=False show_skip_button=True enable_empty_annotation=True show_annotation_history=False organization=1 prompts=None color='#FFFFFF' maximum_annotations=1 is_published=False model_version='FLAIR NER' is_draft=False created_by=UserSimple(id=2, first_name='', last_name='', email='abc@gmail.com', avatar=None) created_at=datetime.datetime(2025, 3, 20, 14, 25, 37, 368466, tzinfo=TzInfo(UTC)) min_annotations_to_start_training=0 start_training_on_annotation_update=False show_collab_predictions=True num_tasks_with_annotations=0 task_number=3 useful_annotation_number=0 ground_truth_number=0 skipped_annotations_number=0 total_annotations_number=0 total_predictions_number=3 sampling='Sequential s

In [13]:
# Create sample tasks
if RECREATE_PROJECT:
    ls.projects.import_tasks(
        id=project.id,
        request=[
            {"text": "Copenhagen Denmark"},
            {"text": "Washington United States"},
            {"text": "Paris France"},
        ]
    )

In [14]:
# Create and connect the ML Backend with the project
flair_model = get_or_create_model(
    ls,
    title='FLAIR NER',
    description='A model to perform NER using Flair',
    url=LABEL_STUDIO_ML_BACKEND_URL,
    project_id=project.id,
    is_interactive=True
)

In [15]:
# Create annotations using ML backend for all the imported tasks 
ls.actions.create(
            id="retrieve_tasks_predictions",
            project=project.id,
            selected_items=ActionsCreateRequestSelectedItemsExcluded(
                all_=True
            )
        )

In [16]:
# Verify the annotations predicted by the ML backend and examine confidence score
for task in ls.tasks.list(project=project.id):
    print(task)

id=385 predictions=[{'id': 90, 'result': [{'from_name': 'label', 'score': 0.7751640975475311, 'to_name': 'text', 'type': 'labels', 'value': {'end': 18, 'labels': ['ORG'], 'start': 0, 'text': 'Copenhagen Denmark'}}], 'model_version': 'Flair-v0.0.1', 'created_ago': '11\xa0minutes', 'score': 0.7751640975475311, 'cluster': None, 'neighbors': None, 'mislabeling': 0.0, 'created_at': '2025-03-20T17:22:01.201286Z', 'updated_at': '2025-03-20T17:22:01.201310Z', 'model': None, 'model_run': None, 'task': 385, 'project': 50}] annotations=[] drafts=[] annotators=[] inner_id=1 cancelled_annotations=0 total_annotations=0 total_predictions=1 completed_at=None file_upload=None storage_filename=None avg_lead_time=None draft_exists=False updated_by=[{'user_id': 2}] data={'text': 'Copenhagen Denmark'} meta={} created_at=datetime.datetime(2025, 3, 20, 14, 25, 37, 390477, tzinfo=TzInfo(UTC)) updated_at=datetime.datetime(2025, 3, 20, 17, 22, 1, 195098, tzinfo=TzInfo(UTC)) is_labeled=False overlap=1.0 comment_